In [52]:
# !pip install openai-agents
# !pip install openai

In [53]:
import os
import json

from pathlib import Path

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "config.py").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from openai import AsyncOpenAI
from agents import Agent, AgentHooks, OpenAIProvider, RunConfig, RunContextWrapper, Runner, function_tool, ModelSettings, AgentHooks, Tool

from core.knowledge_tools import search_knowledge_space


In [54]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY in environment.")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [55]:
MODEL_NAME = "/models/gemma-4-26b-a4b-it"
MODEL_BASE_URL = "http://127.0.0.1:18001/v1"

In [56]:
client = AsyncOpenAI(api_key="null", base_url=MODEL_BASE_URL)

run_config = RunConfig(
    model=MODEL_NAME,
    model_provider=OpenAIProvider(openai_client=client, use_responses=False),
)

knowledge_space = json.loads(Path("../data/knowledge/knowledge_space.json").read_text(encoding="utf-8"))


In [57]:
from typing import Any

class CustomAgentHooks(AgentHooks):
    """Tracks and logs agent lifecycle events for debugging."""
    
    def __init__(self, display_name: str):
        self.event_counter = 0
        self.display_name = display_name

    async def on_start(self, context: RunContextWrapper, agent: Agent) -> None:
        self.event_counter += 1
        print(f"### ({self.display_name}) {self.event_counter}: Agent {agent.name} started")

    async def on_end(self, context: RunContextWrapper, agent: Agent, output: Any) -> None:
        self.event_counter += 1
        print(f"### ({self.display_name}) {self.event_counter}: Agent {agent.name} ended with output {output}")

    async def on_handoff(self, context: RunContextWrapper, agent: Agent, source: Agent) -> None:
        self.event_counter += 1
        print(f"### ({self.display_name}) {self.event_counter}: Agent {source.name} handed off to {agent.name}")

    async def on_tool_start(self, context: RunContextWrapper, agent: Agent, tool: Tool) -> None:
        self.event_counter += 1
        print(f"### ({self.display_name}) {self.event_counter}: Agent {agent.name} started tool {tool.name}")

    async def on_tool_end(self, context: RunContextWrapper, agent: Agent, tool: Tool, result: str) -> None:
        self.event_counter += 1
        print(f"### ({self.display_name}) {self.event_counter}: Agent {agent.name} ended tool {tool.name} with result {result}")

In [58]:
def get_topic_payload(topic_id: str) -> dict | None:
    for topic in knowledge_space.get("topics", []):
        if topic.get("id") == topic_id:
            return topic
    return None

In [59]:
from core.plan_schema import Plan
from core.instructions.planner import PLANNER_PROMPT

planner_agent = Agent(
    name="Planner",
    model=MODEL_NAME,
    instructions=PLANNER_PROMPT,
    output_type=Plan,
    model_settings=ModelSettings(
        temperature=0
    ),
    hooks=CustomAgentHooks("Planner"),
)


In [60]:
@function_tool
def search_topics(query: str) -> str:
    """Search local knowledge graph topics and return candidate topic ids with summaries."""
    result = search_knowledge_space(query)
    return json.dumps(result, ensure_ascii=False, indent=2)


@function_tool
def retrieve_topic(context: RunContextWrapper[CustomAgentHooks], topic_id: str) -> str:
    """Load one exact topic by id from the local knowledge graph and store it in context."""
    topic = get_topic_payload(topic_id)
    if not topic:
        valid_ids = [topic["id"] for topic in knowledge_space.get("topics", [])]
        return f"Unknown topic_id: {topic_id}. Valid ids: {', '.join(valid_ids)}"

    context.context.active_topic_id = topic_id
    context.context.active_topic = topic
    return json.dumps(topic, ensure_ascii=False, indent=2)


@function_tool
async def create_plan(context: RunContextWrapper[CustomAgentHooks], topic_id: str | None = None) -> str:
    """Create a structured learning plan for the current topic or a provided topic id."""
    chosen_topic_id = topic_id or context.context.active_topic_id
    topic = get_topic_payload(chosen_topic_id) if chosen_topic_id else context.context.active_topic
    if not topic:
        return "No topic loaded. Call retrieve_topic first."

    context.context.active_topic_id = topic["id"]
    context.context.active_topic = topic

    planner_input = json.dumps({"topic": topic}, ensure_ascii=False, indent=2)
    result = await Runner.run(
        starting_agent=planner_agent,
        input=planner_input,
        context=context.context,
        run_config=run_config,
        max_turns=1,
    )

    if isinstance(result.final_output, Plan):
        return result.final_output.model_dump_json(indent=2)

    return str(result.final_output)

In [61]:
ASSISTANT_PROMPT = """
You are Leias, a machine learning tutor.

Use only the local knowledge graph tools.

Workflow:
- First call `search_topics` when you need to find the right starting topic.
- Then call `retrieve_topic` with an exact topic id.
- Call `create_plan` only after a topic is retrieved or when the user explicitly asks for a plan.
- Never invent topic ids.
- Never describe fake tool calls in text. Just call the tools.
- Keep the final reply short and useful.
""".strip()

assistant = Agent(
    name="Assistant",
    model=MODEL_NAME,
    instructions=ASSISTANT_PROMPT,
    tools=[search_topics, retrieve_topic, create_plan],
    hooks=CustomAgentHooks("Assistant"),
)

context = CustomAgentHooks()

TypeError: CustomAgentHooks.__init__() missing 1 required positional argument: 'display_name'

In [ ]:
message = "Hello! I am Eduard. I am a novice and want to become Data Scientist. I can write basic Python and want to start machine learning from scratch. What do you recommend?"

result = await Runner.run(
    starting_agent=assistant,
    input=message,
    context=context,
    run_config=run_config
)

print("FINAL RESPONSE:\n")
print(result.final_output)
print("\nACTIVE TOPIC:", context.active_topic_id)

agent start: Assistant
tool finished: Assistant -> search_topics
tool finished: Assistant -> retrieve_topic
agent start: Planner
agent end: Planner
tool finished: Assistant -> create_plan
agent end: Assistant
FINAL RESPONSE:

Hi Eduard! It's great to meet you. Since you already have some Python knowledge, you have a fantastic head start.

To build a solid foundation for Data Science, I recommend we start with the core pillar of machine learning: **Supervised Learning**. 

I have designed a learning plan for you that focuses on:
1.  **The Basics**: Understanding how models use "features" (data) and "labels" (the answers) to learn.
2.  **The Two Main Paths**: Learning to distinguish between **Classification** (predicting categories, like "spam" or "not spam") and **Regression** (predicting numbers, like "house prices").

Shall we dive into the first lesson and explore how these mappings work?

ACTIVE TOPIC: supervised-learning
